# Caderno 13 - Faz expansão de queries usando GPT e Llama

## 1. Chaves de acesso e outras variáveis

In [1]:
from getpass import getpass

GROQ_API = getpass("API Groq")
OPENAI_API = getpass("API OpenAI")

MODELO_LLAMA = "llama3-70b-8192"
MODELO_GPT35 = "gpt-3.5-turbo-0125"
MODELO_GPT4O = "gpt-4o-2024-05-13"

API Groq ········
API OpenAI ········


In [2]:
PASTA_DADOS = './dados/'
PASTA_RESULTADO_CADERNO = f'{PASTA_DADOS}outputs/13_expansao_queries_com_gpt_llama/'

NOME_ARQUIVO_RESULTADO_LLAMA = f'{PASTA_RESULTADO_CADERNO}expansao_queries_llama.pickle'
NOME_ARQUIVO_RESULTADO_GPT35 = f'{PASTA_RESULTADO_CADERNO}expansao_queries_gpt35.pickle'
NOME_ARQUIVO_RESULTADO_GPT4O = f'{PASTA_RESULTADO_CADERNO}expansao_queries_gpt4o.pickle'

## 2. Carrega as queries para aplicar a expansão

In [3]:
import pandas as pd

# A pasta dos JURIS aqui não é a pasta original, e sim o resultado do caderno 1 (os documentos já estão filtrados)
PASTA_JURIS_TCU = f'{PASTA_DADOS}outputs/1_tratamento_juris_tcu/'

# Carrega os arquivos 
def carrega_juris_tcu():
    doc1 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_1.csv', sep='|')
    doc2 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_2.csv', sep='|')
    doc3 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_3.csv', sep='|')
    doc4 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_4.csv', sep='|')
    doc = pd.concat([doc1, doc2, doc3, doc4], ignore_index=True)
    query = pd.read_csv(f'{PASTA_JURIS_TCU}query_tratado.csv', sep='|')
    qrel = pd.read_csv(f'{PASTA_JURIS_TCU}qrel_tratado.csv', sep='|')

    return doc, query, qrel

_, queries, _ = carrega_juris_tcu()

## 3. Define o prompt de sistema que será usado no GPT e no Llama

In [9]:
system_message = """Você é um especialista em sistemas de busca que usam o algoritmo BM25 e está trabalhando com usuários que acessam uma base de dados de jurisprudência do Tribunal de Contas da União. Por se tratar de um sistema de pesquisa léxica, o usuário muitas vezes se depara com o problema de descasamento de vocabulário, pois usa vocabulário em sua query diferente do vocabulário nos documentos indexados. Trata-se de um problema comum, pois o usuário não sabe como o enunciado está escrito.

Para mitigar esse problema, você deverá avaliar a query do usuário e expandi-la com outros termos para mitigar esse problema.

Para isso, você receberá a query do usuário e responderá com a expansão que você acha necessária. A pesquisa será feita com a query original concatenada com a sua expansão fornecida.

A sua resposta deve ser unicamente a expansão da query (NO MÁXIMO 5 PALAVRAS OU TERMOS). Não inclua nenhuma instrução ou explicação sobre sua resposta. Não inclua os termos que já estão presentes na query do usuário.
""".strip()

# Esse formato de mensagens é usado tanto na API da OpenAI quanto na API do Groq
def get_messages(query):
    return [
        { "role": "system", "content": system_message },
        { "role": "user", "content": query}
    ]

## 4. Define chamadas para GPT e Llama

In [10]:
from groq import Groq
from openai import OpenAI

clientGroq = Groq(api_key=GROQ_API)
clientOpenAI = OpenAI(api_key=OPENAI_API)

def completa_llama(query):
    response = clientGroq.chat.completions.create(
        model=MODELO_LLAMA,
        messages=get_messages(query),
        temperature=0,
        max_tokens=1024,
        top_p=1,
        stream=False,
        stop=None,
    )    
    return response.choices[0].message.content

def completa_gpt(query, modelo_gpt):
    response = clientOpenAI.chat.completions.create(
        model=modelo_gpt,
        messages=get_messages(query),
        temperature=1,
        max_tokens=1024,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )
    return response.choices[0].message.content

## 5. Executa as chamadas ao Llama e GPT para toda a base de dados

In [11]:
import os
import pickle

# Objetos que guardarão a expansão das queries
expansao_queries_llama = {}
expansao_queries_gpt35 = {}
expansao_queries_gpt4o = {}

# Verifica se os arquivos já existem. Se já existem, recupera.
if os.path.exists(NOME_ARQUIVO_RESULTADO_LLAMA):
    with open(NOME_ARQUIVO_RESULTADO_LLAMA, 'rb') as f:
        expansao_queries_llama = pickle.load(f)
    print("Resultados do Llama recuperados")

if os.path.exists(NOME_ARQUIVO_RESULTADO_GPT35):
    with open(NOME_ARQUIVO_RESULTADO_GPT35, 'rb') as f:
        expansao_queries_gpt35 = pickle.load(f)
    print("Resultados do GPT-3.5 recuperados")

if os.path.exists(NOME_ARQUIVO_RESULTADO_GPT4O):
    with open(NOME_ARQUIVO_RESULTADO_GPT4O, 'rb') as f:
        expansao_queries_gpt4o = pickle.load(f)
    print("Resultados do GPT-4o recuperados")

In [12]:
import re
from tqdm import tqdm
import time
from formatador import remove_html

# Percorre o dataframe de queries e gera a expasão
for i, row in tqdm(queries.iterrows(), total=len(queries)):
    # Começa a guardar o tempo
    start_time = time.time()
    
    # Extrai a key e o enunciado
    query_key = row.KEY
    query = remove_html(row.TEXT)
    
    # Se já tem o enunciado gerado, passa para o próximo
    if query_key in expansao_queries_llama.keys() and query_key in expansao_queries_gpt35.keys() and query_key in expansao_queries_gpt4o.keys():
        continue
           
    # Gera versões alternativas do enunciado
    expansao_llama = completa_llama(query)
    expansao_gpt35 = completa_gpt(query, MODELO_GPT35)
    expansao_gpt4o = completa_gpt(query, MODELO_GPT4O)
    
    # Salva no mapa
    expansao_queries_llama[query_key] = expansao_llama
    expansao_queries_gpt35[query_key] = expansao_gpt35
    expansao_queries_gpt4o[query_key] = expansao_gpt4o
    
    # Salva em arquivos pickle
    with open(NOME_ARQUIVO_RESULTADO_LLAMA, 'wb') as f:
        pickle.dump(expansao_queries_llama, f)
    
    with open(NOME_ARQUIVO_RESULTADO_GPT35, 'wb') as f:
        pickle.dump(expansao_queries_gpt35, f)

    with open(NOME_ARQUIVO_RESULTADO_GPT4O, 'wb') as f:
        pickle.dump(expansao_queries_gpt4o, f)
    
    # Mede o tempo transcorrido
    elapsed_time = time.time() - start_time
    # Faz um sleep se for necessário
    # (deixar isso habilitado se sair de perto do pc, pois o código não tem try/except)
    # if elapsed_time < 10:
    #    time.sleep(10 - elapsed_time)

100%|████████████████████████████████████████████████████████████████████████████████| 150/150 [06:26<00:00,  2.58s/it]
